# Flickr8k

In [2]:
from datasets import  load_dataset

ds = load_dataset("jxie/flickr8k")
ds

DatasetDict({
    train: Dataset({
        features: ['image', 'caption_0', 'caption_1', 'caption_2', 'caption_3', 'caption_4'],
        num_rows: 6000
    })
    validation: Dataset({
        features: ['image', 'caption_0', 'caption_1', 'caption_2', 'caption_3', 'caption_4'],
        num_rows: 1000
    })
    test: Dataset({
        features: ['image', 'caption_0', 'caption_1', 'caption_2', 'caption_3', 'caption_4'],
        num_rows: 1000
    })
})

# Flickr30k

In [ ]:
!wget https://huggingface.co/datasets/nlphuji/flickr30k/resolve/main/flickr30k-images.zip #~ 4G
!unzip -q flickr30k-images.zip -d flickr30k_images
!rm /kaggle/working/flickr30k-images.zip

In [3]:
import pandas as pd
import ast

df = pd.read_csv('/kaggle/input/datasets/skhalili/flickr30k-annotation/flickr_annotations_30k.csv')
df['raw'] = df['raw'].apply(ast.literal_eval)


train_df = df[df["split"] == "train"].copy()
test_df = df[df["split"] == "test"].copy()
val_df = df[df["split"] == "val"].copy()
IMG_DIR = "/kaggle/working/flickr30k_images/flickr30k-images"

train_df["filename"] = train_df["filename"].apply( lambda x: f"{IMG_DIR}/{x}" )
test_df["filename"] = test_df["filename"].apply( lambda x: f"{IMG_DIR}/{x}" )
val_df["filename"] = val_df["filename"].apply( lambda x: f"{IMG_DIR}/{x}" )


In [4]:
def convert_to_image_level(df):
    grouped = df.groupby(['filename', 'img_id'])['raw'].apply(list).reset_index()

    grouped['raw'] = grouped['raw'].apply(lambda x: sum(x, []))

    captions_df = pd.DataFrame(
        grouped['raw'].tolist(),
        columns=[f'caption_{i}' for i in range(5)]
    )

    result = pd.concat([grouped[['filename', 'img_id']], captions_df], axis=1)
    result = result.rename(columns={'filename': 'image'})

    result = result.set_index('img_id')

    return result
    
train_final = convert_to_image_level(train_df)
val_final   = convert_to_image_level(val_df)
test_final  = convert_to_image_level(test_df)

In [5]:
from datasets import Dataset

train_dataset = Dataset.from_pandas(train_final)
val_dataset   = Dataset.from_pandas(val_final)
test_dataset  = Dataset.from_pandas(test_final)

In [6]:
train_dataset[0]

{'image': '/kaggle/working/flickr30k_images/flickr30k-images/1000092795.jpg',
 'caption_0': 'Two young guys with shaggy hair look at their hands while hanging out in the yard.',
 'caption_1': 'Two young, White males are outside near many bushes.',
 'caption_2': 'Two men in green shirts are standing in a yard.',
 'caption_3': 'A man in a blue shirt standing in a garden.',
 'caption_4': 'Two friends enjoy time spent together.',
 'img_id': 0}

In [7]:
from datasets import Dataset, DatasetDict

ds = DatasetDict({
    "train": Dataset.from_pandas(train_final),
    "validation": Dataset.from_pandas(val_final),
    "test": Dataset.from_pandas(test_final),
})
ds

DatasetDict({
    train: Dataset({
        features: ['image', 'caption_0', 'caption_1', 'caption_2', 'caption_3', 'caption_4', 'img_id'],
        num_rows: 29000
    })
    validation: Dataset({
        features: ['image', 'caption_0', 'caption_1', 'caption_2', 'caption_3', 'caption_4', 'img_id'],
        num_rows: 1014
    })
    test: Dataset({
        features: ['image', 'caption_0', 'caption_1', 'caption_2', 'caption_3', 'caption_4', 'img_id'],
        num_rows: 1000
    })
})

# MS COCO 5K Test Dataset using Karpathy Split

In [ ]:
#https://huggingface.co/datasets/nlphuji/mscoco_2014_5k_test_image_text_retrieval/blob/main/test_5k_mscoco_2014.csv
#!wget https://huggingface.co/datasets/nlphuji/mscoco_2014_5k_test_image_text_retrieval/resolve/main/images_mscoco_2014_5k_test.zip #780M
!unzip /kaggle/working/images_mscoco_2014_5k_test.zip
!rm /kaggle/working/images_mscoco_2014_5k_test.zip

In [8]:
import os

path = "/kaggle/working/images_mscoco_2014_5k_test"

files = os.listdir(path)

print(f"number of files: {len(files)}")
print(files[:20])

number of files: 5000
['COCO_val2014_000000315249.jpg', 'COCO_val2014_000000011081.jpg', 'COCO_val2014_000000311904.jpg', 'COCO_val2014_000000578492.jpg', 'COCO_val2014_000000348881.jpg', 'COCO_val2014_000000248142.jpg', 'COCO_val2014_000000298452.jpg', 'COCO_val2014_000000565331.jpg', 'COCO_val2014_000000058569.jpg', 'COCO_val2014_000000096306.jpg', 'COCO_val2014_000000073333.jpg', 'COCO_val2014_000000078035.jpg', 'COCO_val2014_000000285597.jpg', 'COCO_val2014_000000282251.jpg', 'COCO_val2014_000000073256.jpg', 'COCO_val2014_000000285661.jpg', 'COCO_val2014_000000399921.jpg', 'COCO_val2014_000000556653.jpg', 'COCO_val2014_000000082740.jpg', 'COCO_val2014_000000127516.jpg']


In [9]:
from datasets import Dataset, Image
import ast
import pandas as pd

path = "/kaggle/input/datasets/skhalili/mscoco5k/test_5k_mscoco_2014.csv"

df = pd.read_csv(path)[['filename','raw']]

samples = []

for i, row in df.iterrows():

    captions = row["raw"]
    if isinstance(captions, str):
        captions = ast.literal_eval(captions)

    samples.append({
        "jpg": f"/kaggle/working/images_mscoco_2014_5k_test/{row['filename']}",
        "txt": "\n".join(captions),
    })

dataset = Dataset.from_list(samples)

dataset = dataset.cast_column("jpg", Image())

print(dataset)

Dataset({
    features: ['jpg', 'txt'],
    num_rows: 5000
})


In [10]:
dataset[0]

{'jpg': <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=640x360>,
 'txt': 'A man with a red helmet on a small moped on a dirt road. \nMan riding a motor bike on a dirt road on the countryside.\nA man riding on the back of a motorcycle.\nA dirt path with a young person on a motor bike rests to the foreground of a verdant area with a bridge and a background of cloud-wreathed mountains. \nA man in a red shirt and a red hat is on a motorcycle on a hill side.'}